# 02 — Preprocessing: ASX OHLCV Daily

This notebook applies **controlled transformations** to prepare the ASX OHLCV dataset for downstream analytics and modelling.

Design principles:
- Transformations are explicit, justified, and reproducible
- EDA observations from `01_eda.ipynb` drive decisions here
- No modelling is performed in this notebook


## Scope
This notebook covers:
- Canonical type normalisation
- Missingness and duplicates remediation
- Feature engineering foundations (returns, simple time features)
- Outlier treatment (only if justified)
- Class balance and sampling considerations (only if a target is defined)


In [ ]:
import os
import io
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

from minio import Minio


In [ ]:
@dataclass(frozen=True)
class MinIOConfig:
    endpoint: str = os.environ.get("MINIO_ENDPOINT", "minio:9000")
    access_key: str = os.environ.get("MINIO_ACCESS_KEY", "minioadmin")
    secret_key: str = os.environ.get("MINIO_SECRET_KEY", "minioadmin")
    secure: bool = os.environ.get("MINIO_SECURE", "false").lower() == "true"

CFG = MinIOConfig()

BUCKET = os.environ.get("ASX_BUCKET", "curated")
PREFIX  = os.environ.get("ASX_PREFIX", "tabular/market_ohlcv_daily/exchange=ASX/")
MAX_FILES = int(os.environ.get("ASX_MAX_FILES", "0"))  # 0 means load all

In [ ]:
def minio_client(cfg: MinIOConfig) -> Minio:
    return Minio(cfg.endpoint, access_key=cfg.access_key, secret_key=cfg.secret_key, secure=cfg.secure)

def list_parquet_objects(client: Minio, bucket: str, prefix: str, max_files: int = 0) -> List[str]:
    objs = []
    for obj in client.list_objects(bucket, prefix=prefix, recursive=True):
        name = obj.object_name
        if name.lower().endswith(".parquet"):
            objs.append(name)
            if max_files and len(objs) >= max_files:
                break
    return objs

def read_parquet_object(client: Minio, bucket: str, object_name: str) -> pd.DataFrame:
    resp = client.get_object(bucket, object_name)
    try:
        data = resp.read()
    finally:
        resp.close()
        resp.release_conn()
    return pd.read_parquet(io.BytesIO(data), engine="pyarrow")

def load_parquet_dataset_from_minio(bucket: str, prefix: str, max_files: int = 0) -> Tuple[pd.DataFrame, List[str]]:
    client = minio_client(CFG)
    object_names = list_parquet_objects(client, bucket=bucket, prefix=prefix, max_files=max_files)
    if not object_names:
        return pd.DataFrame(), []
    frames = [read_parquet_object(client, bucket=bucket, object_name=name) for name in object_names]
    return pd.concat(frames, ignore_index=True), object_names


In [ ]:
df_raw, object_names = load_parquet_dataset_from_minio(BUCKET, PREFIX, MAX_FILES)
print(f"Loaded rows: {len(df_raw):,}  |  objects: {len(object_names):,}")
df_raw.head()

## Canonical Type Normalisation

Target conventions:
- `trade_date`: datetime64[ns] (or date) for time operations
- Key identifiers (`exchange`, `ticker`, `vendor`, etc.): string
- OHLCV: numeric (float for prices, integer/float for volume depending on source)


In [ ]:
df = df_raw.copy()

# Strings
for c in ["dataset_id","exchange","ticker","vendor_symbol","vendor","currency","source_object_key","ingest_batch_id","row_hash"]:
    if c in df.columns:
        df[c] = df[c].astype("string")

# Dates
if "trade_date" in df.columns:
    df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce")

# Numerics (coerce errors to NaN)
for c in ["open","high","low","close","adj_close","volume"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df.dtypes

## Duplicates and Key Integrity

Strategy:
- Validate uniqueness on (exchange, ticker, trade_date) if columns exist
- If duplicates exist, apply a deterministic rule (e.g., prefer latest ingest batch) or drop exact duplicates


In [ ]:
key_cols = [c for c in ["exchange","ticker","trade_date"] if c in df.columns]
if len(key_cols) == 3:
    dup_keys = df.duplicated(subset=key_cols).sum()
    print(f"Duplicate key rows: {dup_keys:,}")
else:
    print(f"Key columns present: {key_cols}")

In [ ]:
# Drop full-row duplicates (safe)
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Dropped full-row duplicates: {before-after:,}")

## Missingness Remediation

Typical OHLCV expectations:
- OHLC fields should generally be present if the record exists
- Missingness may indicate upstream issues (late vendor feed, partial loads)

Approach:
- Summarise missingness
- Decide whether to drop rows, forward-fill (rarely appropriate for OHLCV), or leave as-is


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0].head(50)

## Feature Engineering Foundations

These features are broadly useful and low-risk:
- Daily return: `close / lag(close) - 1` (by ticker)
- Log return: `log(close) - log(lag(close))` (by ticker)
- Simple calendar features (optional): day-of-week, month

Notes:
- Ensure sorting by trade_date
- Compute within ticker partitions


In [ ]:
if all(c in df.columns for c in ["ticker","trade_date","close"]):
    df = df.sort_values(["ticker","trade_date"])
    df["close_lag1"] = df.groupby("ticker")["close"].shift(1)
    df["daily_return"] = (df["close"] / df["close_lag1"]) - 1
    df["log_return"] = np.log(df["close"]) - np.log(df["close_lag1"])
df[["ticker","trade_date","close","close_lag1","daily_return","log_return"]].head(20)

## Outlier Treatment Decisions (Optional)

Only apply treatment if you can justify it against downstream objectives.

Typical options:
- Retain (often correct for market data)
- Cap / winsorise (if extreme tails destabilise a model)
- Transform (e.g., log volume)
Default stance here: **retain** and use robust methods downstream, unless there is evidence of data error.


In [ ]:
# Example: log(volume) as a transformation candidate (does not drop rows)
if "volume" in df.columns:
    df["log_volume"] = np.log1p(df["volume"])
    df[["volume","log_volume"]].describe().T

## Target Definition and Sampling Considerations (Deferred)

Sampling (over/under/synthetic) should only be considered **after** defining a target variable.

If you later define a classification target (e.g., `daily_return > 0`), revisit:
- Class distribution
- Potential leakage (future information)
- Train/test split strategy (time-aware split for market data)


In [ ]:
# Optional: Example target definition (commented out by default)
# df["target_up"] = (df["daily_return"] > 0).astype(int)
# df["target_up"].value_counts(normalize=True)

## Outputs

At the end of preprocessing, you typically:
- Persist a cleaned dataset (e.g., Parquet) for modelling notebooks
- Record key decisions (duplicates, missingness, feature definitions)


In [ ]:
# Persist locally (optional) — adjust path as needed
OUT_PATH = os.environ.get("ASX_PREPROCESSED_OUT", "asx_ohlcv_preprocessed.parquet")
df.to_parquet(OUT_PATH, index=False)
print(f"Wrote: {OUT_PATH}  | rows: {len(df):,}  cols: {df.shape[1]:,}")